# Weighted S10000 Out-of-Range Recovery Results

This notebook is the weighted counterpart of the original recovery analysis. It uses the unitary Fourier operator and weighted least squares for the four S10000 source laws, uniform MCS, pure inverse-square sampling, and VDHH. All seven distributions use five sampling ratios and five trials. PSNR, SSIM, and LPIPS receive sampling-ratio sweeps. Outputs stay under `results/weighted/figures/`.

The four shared rates are compared with the original paper results as a complete-pipeline comparison. The `0.025` rate is explicitly labeled weighted-only.


In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd

from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_dir = search_root / 'analyze_results'
    helper_path = helper_dir / 'sd15_recovery_analysis.py'
    if helper_path.exists():
        if str(helper_dir) not in sys.path:
            sys.path.insert(0, str(helper_dir))
        break
    for child in search_root.iterdir():
        if not child.is_dir():
            continue
        helper_dir = child / 'analyze_results'
        helper_path = helper_dir / 'sd15_recovery_analysis.py'
        if helper_path.exists():
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_recovery_analysis.py from the notebook cwd.')

import sd15_recovery_analysis as recovery
recovery = importlib.reload(recovery)

SD15_ROOT = recovery.find_sd15_root(NOTEBOOK_DIR)
WEIGHTED_BASE_TAG = 'weighted/out_of_range/sunset'
SAMPLING_METHODS = list(recovery.WEIGHTED_MAIN_SAMPLING_METHODS)
ALLOWED_SAMPLING_PERC = set(recovery.WEIGHTED_MAIN_RATES)
EXCLUDED_SAMPLING_CONDITIONS = set()
OUTPUT_ROOT = SD15_ROOT / 'results' / 'weighted' / 'figures'

LPIPS_TABLE = recovery.ensure_lpips_metrics(
    SD15_ROOT,
    device='cpu',
)
analysis, COMPLETION_TABLE = recovery.load_weighted_main_analysis(
    SD15_ROOT,
    base_tag=WEIGHTED_BASE_TAG,
    output_root=OUTPUT_ROOT,
    include_partial=True,
)
ROWS = analysis.rows
MEAN_TABLE = analysis.mean_table
ACTIVE_TAG = analysis.active_tag
LOADED_TAGS = analysis.loaded_tags
OUTPUT_DIR = analysis.output_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPLETION_PATH = OUTPUT_DIR / 'weighted_main_completion.csv'
COMPLETION_TABLE.to_csv(COMPLETION_PATH, index=False)

print(f'Active tag: {ACTIVE_TAG}')
print(f'Loaded source tags: {LOADED_TAGS}')
print(f'Loaded {len(ROWS)} / 700 expected run rows.')
lpips_count = int(ROWS['lpips'].notna().sum()) if 'lpips' in ROWS else 0
print(f'Loaded {lpips_count} LPIPS values.')
display(
    COMPLETION_TABLE.groupby(
        ['sampling_method', 'sampling_condition'],
        as_index=False,
    )[['observed', 'expected', 'left']].sum()
)
display(MEAN_TABLE)
if MEAN_TABLE.empty:
    print('No recovery rows found yet. Run the suite first, then rerun this notebook.')


## Metric Curves

This cell calls the shared recovery plotting helpers to export metric curves for `psnr_db`, `ssim`, `lpips`, and `pixel_mae` plus a combined PSNR/SSIM panel for each diffusion/sampling method that has rows. Each subplot fixes a sampling prior, the colored lines compare recovery prompts, and the x-axis is the sampling ratio `m/n` on a log scale.

Curves show the mean over repeats. The shaded region is a 95% normal-approximation confidence interval, computed as mean +/- 1.96 SEM in the plotted metric units. The black dashed reference is the zero-filled inverse FFT baseline when those metrics are present.

In [ ]:
METRIC_OUTPUTS = recovery.export_metric_figures(
    ROWS,
    OUTPUT_DIR,
    combine_sampling_methods=True,
    show=True,
)
METRIC_OUTPUTS


## Recovery Grid

This cell builds the image grids used to inspect reconstruction quality directly. For each sampling prior, it selects one target item and one sampling ratio, then shows the ground truth, the zero-filled inverse FFT baseline, and the best available reconstruction for each recovery prompt.

The "best" reconstruction in each tile is selected from the loaded rows by PSNR first and SSIM second, so the grid is a compact visual counterpart to the metric curves. The cell saves one PDF per sampling prior in `OUTPUT_DIR` and displays the figures inline.

In [ ]:
IMAGE_SAMPLING_PERC = 0.00125

GRID_OUTPUTS = recovery.export_recovery_grids(
    ROWS,
    SD15_ROOT,
    OUTPUT_DIR,
    sampling_method=None,
    sampling_percentage=IMAGE_SAMPLING_PERC,
    show=True,
)
GRID_OUTPUTS


## Complete-Pipeline Comparison with Original Results

The table below compares only the four shared sampling ratios. Differences cannot be attributed to weighting alone because the S10000 secant budget, $\zeta=1/2$ regularization, and unitary Fourier convention also change.


In [ ]:
# This is a complete-pipeline comparison: S10000, zeta, weighting, and FFT normalization all change.
SHARED_PAPER_RATES = {0.00125, 0.0025, 0.005, 0.01}
WEIGHTED_ONLY_RATE = 0.025
ORIGINAL_OUTPUT_ROOT = SD15_ROOT / 'results' / 'unweighted' / 'figures'
original_analysis = recovery.load_recovery_analysis(
    SD15_ROOT,
    tag_group_candidates=recovery.split_tag_group_candidates('unweighted/out_of_range/sunset'),
    sampling_methods=SAMPLING_METHODS,
    allowed_sampling_percentages=SHARED_PAPER_RATES,
    excluded_sampling_conditions=EXCLUDED_SAMPLING_CONDITIONS,
    output_root=ORIGINAL_OUTPUT_ROOT,
)

comparison_keys = [
    'sampling_method',
    'sampling_condition',
    'reconstruction_condition',
    'samp_perc',
]
comparison_metrics = ['psnr_db', 'ssim', 'pixel_mae', 'grain', 'runtime_sec']
comparison_columns = comparison_keys + comparison_metrics
weighted_shared = (
    MEAN_TABLE[
        MEAN_TABLE['samp_perc'].astype(float).isin(SHARED_PAPER_RATES)
    ][comparison_columns].copy()
    if not MEAN_TABLE.empty
    else pd.DataFrame(columns=comparison_columns)
)
original_shared = (
    original_analysis.mean_table[comparison_columns].copy()
    if not original_analysis.mean_table.empty
    else pd.DataFrame(columns=comparison_columns)
)
PIPELINE_COMPARISON = weighted_shared.merge(
    original_shared,
    on=comparison_keys,
    how='outer',
    suffixes=('_weighted', '_original'),
    indicator=True,
)
for metric in comparison_metrics:
    PIPELINE_COMPARISON[f'{metric}_weighted_minus_original'] = (
        PIPELINE_COMPARISON[f'{metric}_weighted'] - PIPELINE_COMPARISON[f'{metric}_original']
    )

RATE_SCOPE = pd.DataFrame(
    {
        'sampling_ratio': sorted(ALLOWED_SAMPLING_PERC),
        'comparison_scope': [
            'shared with original paper' if rate in SHARED_PAPER_RATES else 'weighted-only'
            for rate in sorted(ALLOWED_SAMPLING_PERC)
        ],
    }
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_COMPARISON.to_csv(OUTPUT_DIR / 'weighted_vs_original_shared_rates.csv', index=False)
RATE_SCOPE.to_csv(OUTPUT_DIR / 'rate_scope.csv', index=False)
display(RATE_SCOPE)
display(PIPELINE_COMPARISON)
